# 02 — US DOL forced-labour data and HS6 screening

**Group 4 | Analysis year: 2025 | Target classification: HS2022**

This notebook rebuilds the handoff from preserved official downloads. Raw files are never edited. Read `../README.md` for sources, definitions, and limitations.

**Coverage warning:** this DOL mapping download contains 300 rows (IDs 1–300). It does not map every forced-labour good. Broad-code expansion produces screening candidates, not verified shipment-level findings. The group's SingStat classification has not been checked.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "scripts" / "build_dol_hs6.py").is_file(), "Open from Group 4 or its notebooks folder."
sys.path.insert(0, str(ROOT / "scripts"))
from build_dol_hs6 import build
YEAR = 2025

## 1. Inspect original source data

Read all fields as strings, including country codes and tariff codes. `fl` is the forced-labour listing interval; `cl` and `fcl` are separate categories. The snapshot contains historical intervals, not calendar-year trade values.

In [2]:
raw = ROOT / "data" / "raw" / "us_dol"
core = pd.read_csv(raw / "ILAB_ImportWatch_Core_Data.csv", dtype=str, keep_default_na=False)
mapping = pd.read_csv(raw / "ILAB_ImportWatch_Goods_HS.csv", dtype=str, keep_default_na=False)
print(f"Core records: {len(core)}; source mapping records: {len(mapping)}")
display(core.head())
display(mapping.head())

Core records: 446; source mapping records: 300


,country,country_isocode,region,sector,good,all_exploitations,cl,fl,fcl,country_numericcode
0,Afghanistan,AF,IP,Manufacturing,Bricks,CL FL FCL,2009-,2011-,2011-,5310
1,Afghanistan,AF,IP,Manufacturing,Carpets,CL,2009-,,,5310
2,Afghanistan,AF,IP,Mining,Coal,CL,2012-,,,5310
3,Afghanistan,AF,IP,Agriculture,Poppies,CL,2009-,,,5310
4,Afghanistan,AF,IP,Mining,Salt,CL,2018-,,,5310


,id,good,hts_code
0,1,Alcoholic Beverages,2203
1,10,Bamboo,4402.1
2,100,Coconuts,0801.11.0000
3,101,Coconuts,0801.12.0000
4,102,Coconuts,0801.19.01


## 2. Build HS6 candidates and validate outputs

The script verifies source hashes, filters active `fl` intervals, matches exact DOL good names, validates code prefixes against UN HS2022, and records unresolved rows. Longer HTS codes collapse to HS6; shorter prefixes expand to existing six-digit codes and are flagged for scope review. It also creates a deduplicated country/HS6 table to prevent duplication of trade values in later joins.

In [3]:
summary = build(YEAR)
print(json.dumps(summary, indent=2))

{
  "analysis_year": 2025,
  "hs_revision": "HS2022",
  "raw_core_rows": 446,
  "raw_mapping_rows": 300,
  "hs2022_reference_codes": 5613,
  "active_fl_country_good_pairs": 119,
  "mapped_country_good_pairs": 91,
  "unmapped_country_good_pairs": 28,
  "candidate_detail_rows": 3163,
  "unique_country_hs6_candidates": 3156,
  "unique_hs6_candidates": 832,
  "review_counts": {
    "broad_prefix_expansion_needs_scope_review": 124,
    "no_mapping_in_download": 28,
    "no_valid_HS2022_match": 2
  },
  "unmapped_goods": [
    "Hair products",
    "Pornography",
    "Stones",
    "Sugarcane",
    "Sunflowers",
    "Tantalum ore (coltan)",
    "Teak",
    "Textiles",
    "Textiles (hand-woven)",
    "Thread/Yarn",
    "Tilapia (fish)",
    "Timber",
    "Tin ore (cassiterite)",
    "Tobacco",
    "Tomato Products",
    "Tomatoes",
    "Toys",
    "Tungsten ore (wolframite)",
    "Wheat"
  ],
  "warning": "Partial historical DOL snapshot; screening candidates, not a complete validated risk uni

## 3. Review gaps before joining trade data

Unmapped records remain in the coverage and review files. A country-good pair with candidates can still have missing mappings or overbroad coverage. Absence of a match is not evidence of zero risk.

In [4]:
out = ROOT / "data" / "processed"
coverage = pd.read_csv(out / f"dol_mapping_coverage_{YEAR}.csv", dtype={"country_isocode": str})
review = pd.read_csv(out / f"dol_mapping_review_{YEAR}.csv", dtype=str, keep_default_na=False)
display(coverage.groupby("mapping_coverage").size().rename("country_good_pairs").to_frame())
display(review.groupby("reason").size().rename("review_items").to_frame())
display(coverage.loc[coverage["mapping_coverage"].eq("unmapped"), ["country", "good"]])

,country_good_pairs
mapping_coverage,
has_candidates,91
unmapped,28


,review_items
reason,
broad_prefix_expansion_needs_scope_review,124
no_mapping_in_download,28
no_valid_HS2022_match,2


,country,good
9,Bolivia,Sugarcane
14,Brazil,Sugarcane
15,Brazil,Timber
28,Burma,Sugarcane
29,Burma,Sunflowers
30,Burma,Teak
43,China,Hair products
46,China,Textiles
47,China,Thread/Yarn
48,China,Tomato Products


## 4. Load the country/HS6 screening table

Use ISO2 origin/producing-country codes and six-character HS2022 strings. Confirm the SingStat classification first. The detail file preserves provenance and may repeat a trade key; the screening file below is unique by country/HS6. An `exact_hs6` label confirms only the source code length and existence in HS2022, not historical revision equivalence.

In [5]:
screening = pd.read_csv(out / f"dol_country_hs6_screening_{YEAR}.csv", dtype={"country_isocode": str, "hs6": str})
assert screening["hs6"].str.fullmatch(r"\d{6}").all()
assert not screening.duplicated(["country_isocode", "hs6"]).any()
assert len(screening) == summary["unique_country_hs6_candidates"]
assert len(coverage) == summary["active_fl_country_good_pairs"]
assert screening["hs6"].str.startswith("0").any(), "Leading-zero codes must survive."
display(screening.head(10))
print("Validation passed: source hashes, valid HS2022 codes, leading zeroes, unique country/HS6 keys, and coverage reconciliation.")

,country_isocode,country,hs6,hs_revision,analysis_year,dol_goods,mapping_methods,has_broad_expansion,screening_candidate
0,AF,Afghanistan,681011,HS2022,2025,Bricks,exact_hs6,0,1
1,AF,Afghanistan,690100,HS2022,2025,Bricks,broad_prefix_expansion,1,1
2,AF,Afghanistan,690210,HS2022,2025,Bricks,broad_prefix_expansion,1,1
3,AF,Afghanistan,690220,HS2022,2025,Bricks,hts_prefix_to_hs6,0,1
4,AF,Afghanistan,690290,HS2022,2025,Bricks,hts_prefix_to_hs6,0,1
5,AF,Afghanistan,690410,HS2022,2025,Bricks,broad_prefix_expansion,1,1
6,AF,Afghanistan,690490,HS2022,2025,Bricks,broad_prefix_expansion,1,1
7,AO,Angola,710210,HS2022,2025,Diamonds,broad_prefix_expansion,1,1
8,AO,Angola,710221,HS2022,2025,Diamonds,broad_prefix_expansion,1,1
9,AO,Angola,710229,HS2022,2025,Diamonds,broad_prefix_expansion,1,1


Validation passed: source hashes, valid HS2022 codes, leading zeroes, unique country/HS6 keys, and coverage reconciliation.


## Handoff

Send the ZIP containing the whole Group 4 folder. Your teammate can use the included CSVs immediately or rerun the notebook in their own `.venv`. No Git or API key is required.

Before analysis: resolve entries in the review file, confirm the trade-data classification and origin-country definition, and assess the incomplete historical source coverage. See the README for a `many_to_one` join example. Do not automatically replace unmatched candidate flags with zero.